# 3D Computer Vision Project — Demo

Minimal end-to-end demo calling the two assignment-level functions:

- `calibrate(imgs)` — intrinsics + scene pose from checkerboard images.
- `move_block(blocks, img, calib)` — command string for picking and placing cubes.

For a full visualized pipeline walkthrough (checkerboard detection, object detection, animations), see [walktrough.ipynb](walktrough.ipynb).

## Setup

In [7]:
import os, glob
from PIL import Image
from pathlib import Path

from main import calibrate, move_block

INTRINSIC_CALIB_DIR = Path(os.path.join("test-images", "intrinsic_calibration"))
SCENE_DIR = Path(os.path.join("test-images", "scene6"))
SCENE_IMAGES_DIR = SCENE_DIR / "images"
SCENE_CALIB_DIR = SCENE_DIR / "calibration"

BLOCKS = ['red', 'green', 'blue']

# Real-world calibration of robot motion. Defaults are no-ops; tune if the
# robot under/overshoots or turns the wrong way (e.g. TURN_SCALE=-1.0).
DRIVE_SCALE, DRIVE_BIAS = 1.0, 0.0
TURN_SCALE,  TURN_BIAS  = -1.0, 0.0

intrinsic_calib_images = [Image.open(p) for p in sorted(glob.glob(os.path.join(INTRINSIC_CALIB_DIR, '*.png')))]
extrinsic_calib_image = Image.open(sorted(glob.glob(os.path.join(SCENE_CALIB_DIR, '*.png')))[0])
scene_image  = Image.open(sorted(glob.glob(os.path.join(SCENE_IMAGES_DIR, '*.png')))[0])

print(f'Loaded {len(intrinsic_calib_images)} intrinsic calibration image(s), ' \
      f"1 extrinsic calibration image and 1 scene image")

Loaded 13 intrinsic calibration image(s), 1 extrinsic calibration image and 1 scene image


## 1. Calibrate

In [8]:
calib = calibrate(intrinsic_calib_images, extrinsic_calib_image)
K = calib['K']
print(f"\nfx={K[0,0]:.1f}, fy={K[1,1]:.1f}, cx={K[0,2]:.1f}, cy={K[1,2]:.1f}")

[compute_intrinsics] Zhang's K:
[[ 6.5e+03 -3.6e+03 -6.2e+03]
 [ 0.0e+00  3.2e+02 -3.2e+03]
 [ 0.0e+00  0.0e+00  1.0e+00]]
[compute_intrinsics] Fallback to default K estimate
[compute_intrinsics] RMS reprojection error: 3.6187 px (13 views)
[compute_extrinsics] scene-pose RMS: 1.056 px

fx=7927.5, fy=7927.5, cx=2710.1, cy=1036.8


## 2. Plan moves

In [9]:
BLOCKS = ["red"]

commands = move_block(BLOCKS, scene_image, calib,
                      drive_scale=DRIVE_SCALE, drive_bias=DRIVE_BIAS,
                      turn_scale=TURN_SCALE,  turn_bias=TURN_BIAS)

print(f'Commands for blocks={BLOCKS}:')
print(commands)

Commands for blocks=['red']:
turn(124.9); go(10.0); grab(); turn(-99.5); go(25.0); let_go()
